In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import ast
import os



# matplotlib settings
# --- Publication style settings ---
plt.rcParams.update({
    # Font
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "text.usetex": False,

    # Editable, Affinity-compatible font embedding
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    # Figure
    "figure.figsize": (6, 4),
    "figure.dpi": 150,

    # Axes
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.linewidth": 1.2,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.7,

    # Ticks
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,

    # Lines
    "lines.linewidth": 2,
    "lines.markersize": 6,

    # Legend
    "legend.fontsize": 10,
    "legend.frameon": False
})

# functions to calculate statistics
def RMSE(actual, predicted):
    
    # Convert inputs to numpy arrays 
    actual = np.array(actual)
    predicted = np.array(predicted)

    
    # Calculate the squared differences
    squared_diff = (actual - predicted) ** 2
    
    # Compute the mean of squared differences
    mean_squared_diff = np.mean(squared_diff)
    
    # Take the square root of the mean squared difference
    return np.sqrt(mean_squared_diff)

def find_xtc_filenames(path_to_dir, suffix=".xtc"):
    filenames = os.listdir(path_to_dir)
    return [filename for filename in filenames if filename.endswith(suffix)]

def compute_free_energy(Q, nbins=30, unfolded_cutoff=0.1, wopt=None):
    k = 0.008314
    T = 303.0
    kT = k * T

    
    if wopt is None:
        wopt = np.ones(Q.shape[0])/Q.shape[0]
        

    # Histograms
    def compute_FEL(Q, weights=None):
        counts, bins = np.histogram(Q, bins=nbins, weights=weights)
        centers = 0.5 * (bins[1:] + bins[:-1])
        p = counts / np.sum(counts)
        p[p <= 0] = np.nan
        fel = -np.log(p) * kT
        fel -= np.nanmin(fel)
        return centers, fel * 0.239005736

    centers_prior, fel_prior = compute_FEL(Q)
    centers_post, fel_post = compute_FEL(Q, weights=wopt)

    # Energetics 1: completely unfolded state relative to native state (comparable to what HDX probes)
    unfolded = np.where(Q < unfolded_cutoff)[0]
    # get cutoff for folded state from FEL
    min_Q_idx = np.where(fel_prior ==np.min(fel_prior))[0]
    minQ = centers_prior[min_Q_idx]
    native_cutoff = 0.7 * minQ
    folded = np.where(Q > native_cutoff)[0]

    dG_prior = -kT * np.log(len(folded) / len(unfolded)) * 0.239005736
    dG_post = -kT * np.log(np.sum(wopt[folded]) / np.sum(wopt[unfolded])) * 0.239005736

    # Energetics 2: native state relative to the rest (including potential intermediates, comparable to what cDNA proteolysis probes)
    # criteria: the main energy barrier between native and unfolded must be overcome --> cutoff decided as the max free energy between Q 0.2 and 0.8
    mask = (centers_prior > 0.2) & (centers_prior < 0.8)
    max_Q_idx = np.where(fel_prior ==np.max(fel_prior[mask]))[0][0]
    native_cutoff2 = centers_prior[max_Q_idx]
    folded = np.where(Q > native_cutoff2)[0]
    unfolded = np.where(Q < native_cutoff2)[0]

    dG_prior2 = -kT * np.log(len(folded) / len(unfolded)) * 0.239005736
    dG_post2 = -kT * np.log(np.sum(wopt[folded]) / np.sum(wopt[unfolded])) * 0.239005736

    return centers_prior, fel_prior, centers_post, fel_post, dG_prior, dG_post, dG_prior2, dG_post2, native_cutoff, native_cutoff2

def find_elbow(x, y):
    # Convert to numpy arrays
    x = np.array(x)
    y = np.array(y)

    # Sort x (and y) if needed, but keep track of original indices
    if not np.all(np.diff(x) >= 0):
        idx_sort = np.argsort(x)
        x_sorted = x[idx_sort]
        y_sorted = y[idx_sort]
    else:
        idx_sort = np.arange(len(x))
        x_sorted, y_sorted = x, y

    # Line between first and last point
    p1 = np.array([x_sorted[0], y_sorted[0]])
    p2 = np.array([x_sorted[-1], y_sorted[-1]])
    line_vec = p2 - p1
    line_vec_norm = line_vec / np.linalg.norm(line_vec)

    # Distances from each point to the line
    vec_from_p1 = np.vstack([x_sorted - p1[0], y_sorted - p1[1]]).T
    proj_len = np.dot(vec_from_p1, line_vec_norm)
    proj_point = np.outer(proj_len, line_vec_norm) + p1
    dist = np.linalg.norm(vec_from_p1 - (proj_point - p1), axis=1)

    # Elbow = point with maximum distance
    idx_elbow_sorted = np.argmax(dist)
    idx_elbow_original = idx_sort[idx_elbow_sorted]

    return idx_elbow_original, x[idx_elbow_original], y[idx_elbow_original]


proteins = ['Ubiquitin']
protein = proteins[0]
ensemble_master_dir = 'DESRES_FOLDING_DATA'
outdir = './plots_Ubq_desres/'

In [2]:
# secondary structure

import mdtraj as md

# Function to calculate per-residue helicity per frame
def dssp_to_helicity(dssp_lines):
    """
    Convert DSSP strings into a binary helicity matrix.
    
    Parameters
    ----------
    dssp_lines : list of str
        Each entry corresponds to one frame.
        Each string is the DSSP assignment per residue for that frame.
    
    Returns
    -------
    helicity : np.ndarray
        Array of shape (n_frames, n_residues).
        1 = helix ('H', 'G', 'I'), 0 = non-helix.
    """
    helix_codes = {'H', 'G', 'I'}
    
    n_frames = len(dssp_lines)
    n_residues = len(dssp_lines[0])
    
    helicity = np.zeros((n_frames, n_residues), dtype=int)
    
    for i, line in enumerate(dssp_lines):
        helicity[i, :] = [1 if c in helix_codes else 0 for c in line]
    
    return helicity

# Function to calculate per-residue helicity per frame
def dssp_to_strand(dssp_lines):
    """
    same thing for beta strand
    """
    strand_codes = {'E', 'B'}
    
    n_frames = len(dssp_lines)
    n_residues = len(dssp_lines[0])
    
    strandicity = np.zeros((n_frames, n_residues), dtype=int)
    
    for i, line in enumerate(dssp_lines):
        strandicity[i, :] = [1 if c in strand_codes else 0 for c in line]
    
    return strandicity

# comparison with secondary structure / helicity
path = '../../DESRES_FOLDING_DATA/Ubiquitin/'

traj_names = np.sort(find_xtc_filenames(path))

dssp_all = []

for traj in traj_names:
    print(traj)
    u = md.load(
        path + f'./{traj}',
        top=path + f'./protein.pdb'
    )

    heavyatoms = u.topology.select('not element H')
    u = u.atom_slice(heavyatoms)

    tmpdssp = md.compute_dssp(u)
    dssp_all.append(tmpdssp)

dssp = np.concatenate(dssp_all, axis=0)
np.save(path+'dssp_all_trajs.npy', dssp)



pnas2013-native-1-protein-000.xtc
pnas2013-native-1-protein-001.xtc
pnas2013-native-1-protein-002.xtc
pnas2013-native-1-protein-003.xtc
pnas2013-native-1-protein-004.xtc
pnas2013-native-1-protein-005.xtc
pnas2013-native-1-protein-006.xtc
pnas2013-native-1-protein-007.xtc
pnas2013-native-1-protein-008.xtc
pnas2013-native-1-protein-009.xtc
pnas2013-native-1-protein-010.xtc
pnas2013-native-1-protein-011.xtc
pnas2013-native-1-protein-012.xtc
pnas2013-native-1-protein-013.xtc
pnas2013-native-1-protein-014.xtc
pnas2013-native-1-protein-015.xtc
pnas2013-native-1-protein-016.xtc
pnas2013-native-1-protein-017.xtc
pnas2013-native-1-protein-018.xtc
pnas2013-native-1-protein-019.xtc
pnas2013-native-1-protein-020.xtc
pnas2013-native-1-protein-021.xtc
pnas2013-native-1-protein-022.xtc
pnas2013-native-1-protein-023.xtc
pnas2013-native-1-protein-024.xtc
pnas2013-native-1-protein-025.xtc
pnas2013-native-1-protein-026.xtc
pnas2013-native-1-protein-027.xtc
pnas2013-native-1-protein-028.xtc
pnas2013-nativ

In [6]:
# RMS deviation of native backbone hydrogen-bond distances
# for beta-strand pairs 5-1 and 5-3

import mdtraj as md
import numpy as np

path = '../../DESRES_FOLDING_DATA/Ubiquitin/'
native_file = path + 'exp_native_renum.pdb'
topology_file = path + 'protein.pdb'

traj_names = np.sort(find_xtc_filenames(path))

# ------------------------------------------------------------------
# Load native reference
# ------------------------------------------------------------------

native = md.load(native_file)

# ------------------------------------------------------------------
# Identify beta strands from native DSSP
# require at least 4 contiguous E residues to count as a strand
# ------------------------------------------------------------------

dssp_native = md.compute_dssp(native, simplified=True)[0]

min_strand_length = 2

strand_residues = []
current_strand = []

for i, ss in enumerate(dssp_native):
    if ss == 'E':
        current_strand.append(i)
    else:
        if len(current_strand) >= min_strand_length:
            strand_residues.append(current_strand)
        current_strand = []

if len(current_strand) >= min_strand_length:
    strand_residues.append(current_strand)

print(f'Found {len(strand_residues)} beta strands:')

for i, residues in enumerate(strand_residues):
    resseq = [native.topology.residue(r).resSeq for r in residues]
    print(f'Strand {i+1}: residues {resseq}')

assert len(strand_residues) >= 5, 'Fewer than 5 beta strands were identified.'

strand1 = set(strand_residues[0])
strand3 = set(strand_residues[2])
strand5 = set(strand_residues[4])

print(strand1)
print(strand3)
print(strand5)

# ------------------------------------------------------------------
# Find native backbone hydrogen bonds
# Baker-Hubbard returns donor-heavy, donor-H, acceptor-heavy atom indices
# ------------------------------------------------------------------

hbonds_native = md.baker_hubbard(native, periodic=False)

selected_hbonds = []

for donor, hydrogen, acceptor in hbonds_native:
    donor_atom = native.topology.atom(donor)
    hydrogen_atom = native.topology.atom(hydrogen)
    acceptor_atom = native.topology.atom(acceptor)

    donor_res = donor_atom.residue.index
    acceptor_res = acceptor_atom.residue.index

    # backbone amide N-H donor and backbone carbonyl O acceptor only
    if donor_atom.name != 'N' or acceptor_atom.name not in ('O', 'OXT'):
        continue

    # strand 5 <-> strand 1
    pair_51 = (donor_res in strand5 and acceptor_res in strand1) or (donor_res in strand1 and acceptor_res in strand5)

    # strand 5 <-> strand 3
    pair_53 = (donor_res in strand5 and acceptor_res in strand3) or (donor_res in strand3 and acceptor_res in strand5)

    if pair_51 or pair_53:
        selected_hbonds.append((donor, hydrogen, acceptor))

selected_hbonds = np.asarray(selected_hbonds, dtype=int)

print(f'\nSelected {len(selected_hbonds)} native backbone hydrogen bonds:')

for donor, hydrogen, acceptor in selected_hbonds:
    donor_atom = native.topology.atom(donor)
    acceptor_atom = native.topology.atom(acceptor)
    print(f'{donor_atom.residue} N-H  ->  {acceptor_atom.residue} O')

assert len(selected_hbonds) > 0, 'No strand 5-1 or strand 5-3 native hydrogen bonds were found.'

# ------------------------------------------------------------------
# Native H...O distances
# ------------------------------------------------------------------

ho_pairs = selected_hbonds[:, [1, 2]]
native_distances = md.compute_distances(native, ho_pairs, periodic=False)[0]

print('\nNative H...O distances (nm):')

for (donor, hydrogen, acceptor), distance in zip(selected_hbonds, native_distances):
    donor_atom = native.topology.atom(donor)
    acceptor_atom = native.topology.atom(acceptor)
    print(f'{donor_atom.residue} -> {acceptor_atom.residue}: {distance:.3f} nm')

# ------------------------------------------------------------------
# Calculate RMS deviation from native H-bond distances
# for every trajectory frame
# ------------------------------------------------------------------

hbond_distance_rmsd_all = []

for traj in traj_names:
    print(traj)

    u = md.load(path + traj, top=topology_file)

    distances = md.compute_distances(u, ho_pairs, periodic=False)
    hbond_distance_rmsd = np.sqrt(np.mean((distances - native_distances[None, :])**2, axis=1))

    hbond_distance_rmsd_all.append(hbond_distance_rmsd)

# ------------------------------------------------------------------
# Concatenate trajectories and save
# units are nm
# ------------------------------------------------------------------

hbond_distance_rmsd_all = np.concatenate(hbond_distance_rmsd_all)

print('\nTotal frames:', len(hbond_distance_rmsd_all))
print('Mean H-bond distance RMSD:', np.mean(hbond_distance_rmsd_all), 'nm')

np.save(path + 'hbond_distance_rmsd_all_trajs.npy', hbond_distance_rmsd_all)

Found 5 beta strands:
Strand 1: residues [2, 3, 4, 5, 6, 7]
Strand 2: residues [12, 13, 14, 15, 16]
Strand 3: residues [41, 42, 43, 44, 45]
Strand 4: residues [48, 49]
Strand 5: residues [66, 67, 68, 69, 70, 71]
{1, 2, 3, 4, 5, 6}
{40, 41, 42, 43, 44}
{65, 66, 67, 68, 69, 70}

Selected 7 native backbone hydrogen bonds:
LYS6 N-H  ->  LEU67 O
ARG42 N-H  ->  VAL70 O
ILE44 N-H  ->  HIS68 O
LEU67 N-H  ->  PHE4 O
HIS68 N-H  ->  ILE44 O
LEU69 N-H  ->  LYS6 O
VAL70 N-H  ->  ARG42 O

Native H...O distances (nm):
LYS6 -> LEU67: 0.192 nm
ARG42 -> VAL70: 0.196 nm
ILE44 -> HIS68: 0.183 nm
LEU67 -> PHE4: 0.186 nm
HIS68 -> ILE44: 0.180 nm
LEU69 -> LYS6: 0.193 nm
VAL70 -> ARG42: 0.183 nm
pnas2013-native-1-protein-000.xtc
pnas2013-native-1-protein-001.xtc
pnas2013-native-1-protein-002.xtc
pnas2013-native-1-protein-003.xtc
pnas2013-native-1-protein-004.xtc
pnas2013-native-1-protein-005.xtc
pnas2013-native-1-protein-006.xtc
pnas2013-native-1-protein-007.xtc
pnas2013-native-1-protein-008.xtc
pnas2013-nati